# 05 — Closed-Loop Control
## LQR Rescue: Computing the Minimum-Energy Perturbation That Saves a Failing Trajectory

---

This is the bridge from observation to intervention — from neuroscience to neuroengineering.

In Modules 2-4 you showed that certain latent trajectories are geometrically unstable (high tangling, collapsing principal angles, eigenvalues drifting from unit circle). Now:

> **Given a trajectory heading toward failure, what is the minimum-energy perturbation to the latent state that would return it to the correct attractor?**

This is exactly the **Linear Quadratic Regulator (LQR)** problem.

**Why this matters for BCI:**
The control signal $u_t^*$ you compute here is a *stimulation target* in latent space. It says: "apply this perturbation to the neural population state, and the failing WM trajectory will stabilize." In a real closed-loop BCI, $u_t^*$ would be inverted through the electrode-tissue model to generate actual stimulation currents.

---

**Read before this module:**
- Stengel — *Optimal Control and Estimation* — Ch. 4 (LQR)
- Shanechi (2019) — *Brain-machine interface control algorithms* — Nature Neuroscience

In [ ]:
import numpy as np
import scipy.linalg
import matplotlib.pyplot as plt
import sys
from pathlib import Path

sys.path.insert(0, str(Path('../../scripts').resolve()))
plt.rcParams.update({'font.size': 11, 'figure.dpi': 110})

---
## 1. LQR Theory — What You Must Derive

**Setup:** You have a linear system with control input:
$$z_{t+1} = A z_t + B u_t$$

You want to minimize the cost over a finite horizon T:
$$J = \sum_{t=0}^{T} \left( z_t^\top Q_c z_t + u_t^\top R_c u_t \right)$$

Where:
- $Q_c$ penalizes **deviation from target state** (state cost matrix)
- $R_c$ penalizes **control energy** (input cost matrix)
- Larger $Q_c/R_c$ ratio → accepts more stimulation energy to hit the target
- Smaller ratio → accepts more state error to save energy

**The solution (Bellman equation → Riccati equation):**

Solve the Discrete Algebraic Riccati Equation (DARE) for P:
$$P = Q_c + A^\top P A - A^\top P B (R_c + B^\top P B)^{-1} B^\top P A$$

Then the optimal gain: $K = (R_c + B^\top P B)^{-1} B^\top P A$

And optimal control: $u_t^* = -K z_t$

### ✏️ Exercise — Derive LQR from Scratch

1. Write the Bellman equation for this problem: $V_t(z) = \min_{u_t} [z^\top Q_c z + u^\top R_c u + V_{t+1}(Az + Bu)]$
2. Assume $V_t(z) = z^\top P_t z$ (quadratic cost-to-go) and solve for $P_t$
3. Show that the recursion for $P_t$ is the Riccati equation
4. Show that the optimal control $u_t^* = -K z_t$ is linear

Write this in `notes/lqr_riccati_derivation.md`. This derivation is your proof that you understand optimal control, not just its output.

In [ ]:
def design_lqr(A, B, Q_c, R_c):
    """
    Solve the discrete-time LQR problem.

    Returns:
        K : (m, n) optimal gain matrix
        P : (n, n) solution to the DARE

    scipy.linalg.solve_discrete_are(A, B, Q, R) solves:
    P = Q + AᵀPA - AᵀPB(R + BᵀPB)⁻¹BᵀPA
    """
    P = scipy.linalg.solve_discrete_are(A, B, Q_c, R_c)
    K = np.linalg.solve(R_c + B.T @ P @ B, B.T @ P @ A)
    return K, P


def simulate_lqr_rescue(A, B, K, z0_fail, z_target,
                         T=100, noise_std=0.01):
    """
    Simulate the LQR-controlled trajectory: does it converge to z_target?

    Args:
        z0_fail  : (n,) starting state (the pre-failure latent state)
        z_target : (n,) target state (the centroid of successful trials)
        noise_std: process noise (biological variability)

    Returns:
        z_free      : (T, n) trajectory WITHOUT control
        z_controlled: (T, n) trajectory WITH LQR control
        u_history   : (T, m) control inputs over time
    """
    n = len(z0_fail)
    rng = np.random.default_rng(0)

    # Error state: e_t = z_t - z_target
    # Controlled: e_{t+1} = (A - BK) e_t + noise
    A_cl = A - B @ K   # closed-loop A matrix (should be stable)

    # Free (uncontrolled)
    z_free = np.zeros((T, n))
    z_free[0] = z0_fail
    for t in range(T-1):
        z_free[t+1] = A @ z_free[t] + rng.standard_normal(n) * noise_std

    # Controlled
    e = np.zeros((T, n))
    u_hist = np.zeros((T, B.shape[1]))
    e[0] = z0_fail - z_target
    for t in range(T-1):
        u_hist[t] = -K @ e[t]       # u* = -K * error
        e[t+1] = A_cl @ e[t] + rng.standard_normal(n) * noise_std
    z_controlled = e + z_target     # add back target offset

    return z_free, z_controlled, u_hist


print('LQR functions defined.')

In [ ]:
# ─── Toy demonstration: LQR on a 2D WM-like system ───────────────────────────
# Use A_wm from Module 0a (or define here)

theta = 0.15
A_wm  = 0.97 * np.array([[np.cos(theta), -np.sin(theta)],
                           [np.sin(theta),  np.cos(theta)]])  # slight decay

B = np.eye(2)  # identity: full access to state (unrealistic but instructive)

# State cost: penalize large deviation from target
Q_c = np.eye(2) * 10  # Q_c = 10 I: strong state penalty
R_c = np.eye(2) * 1   # R_c = 1 I: weak control penalty

K, P = design_lqr(A_wm, B, Q_c, R_c)
print(f'LQR gain K:\n{K.round(4)}')
print(f'\nClosed-loop eigenvalues: {np.abs(np.linalg.eigvals(A_wm - B @ K)).round(4)}')
print('(Should all be < 1 = stable)')

# Simulate rescue: failing state starts away from target
z_target = np.array([0.8, 0.0])   # target: maintain item near this attractor
z0_fail  = np.array([0.0, -0.8])  # failure start: drifted to wrong attractor

z_free, z_ctrl, u_hist = simulate_lqr_rescue(A_wm, B, K, z0_fail, z_target, T=80)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# State-space view
ax = axes[0]
ax.plot(z_free[:,0], z_free[:,1], 'crimson', lw=1.5, label='Uncontrolled (failure)')
ax.plot(z_ctrl[:,0], z_ctrl[:,1], 'royalblue', lw=2, label='LQR controlled')
ax.scatter(*z_target, color='gold', s=120, zorder=5, label='Target state', marker='*')
ax.scatter(*z0_fail,  color='k',    s=60,  zorder=5, label='Start (failure)')
theta_ref = np.linspace(0, 2*np.pi, 100)
ax.plot(np.cos(theta_ref), np.sin(theta_ref), 'k--', alpha=0.2, lw=1)
ax.set_xlabel('Latent dim 1'); ax.set_ylabel('Latent dim 2')
ax.set_title('LQR Rescue in State Space')
ax.set_aspect('equal'); ax.legend(fontsize=9)

# Distance to target over time
ax = axes[1]
dist_free = np.linalg.norm(z_free - z_target, axis=1)
dist_ctrl = np.linalg.norm(z_ctrl - z_target, axis=1)
ax.plot(dist_free, 'crimson', lw=2, label='Uncontrolled')
ax.plot(dist_ctrl, 'royalblue', lw=2, label='LQR controlled')
ax.set_xlabel('Time steps')
ax.set_ylabel('Distance to target state')
ax.set_title('Convergence to Target')
ax.legend()
ax2 = ax.twinx()
ax2.plot(np.linalg.norm(u_hist, axis=1), 'gray', lw=1, linestyle='--', label='Control effort ‖u‖')
ax2.set_ylabel('Control effort ‖u‖', color='gray')
ax2.legend(loc='lower right', fontsize=9)

plt.suptitle(f'LQR Rescue — Q/R ratio = {Q_c[0,0]}/{R_c[0,0]}', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Energy-accuracy tradeoff: sweep R_c ─────────────────────────────────────
# The Q/R ratio controls: accept more distance to target OR use more energy.
# In a real BCI: R_c represents safety constraints on stimulation amplitude.

R_vals = [0.01, 0.1, 1.0, 10.0, 100.0]
final_distances = []
total_energies  = []

for R_val in R_vals:
    R_c_sweep = np.eye(2) * R_val
    K_s, _    = design_lqr(A_wm, B, Q_c, R_c_sweep)
    _, z_c, u_c = simulate_lqr_rescue(A_wm, B, K_s, z0_fail, z_target, T=80)
    final_distances.append(np.linalg.norm(z_c[-1] - z_target))
    total_energies.append(np.sum(u_c**2))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].semilogx(R_vals, final_distances, 'steelblue', marker='o', lw=2)
axes[0].set_xlabel('R_c (control penalty)')
axes[0].set_ylabel('Final distance to target')
axes[0].set_title('State Error vs Control Penalty')

axes[1].loglog(total_energies, final_distances, 'crimson', marker='o', lw=2)
for i, R in enumerate(R_vals):
    axes[1].annotate(f'R={R}', (total_energies[i], final_distances[i]),
                     textcoords='offset points', xytext=(5,5), fontsize=8)
axes[1].set_xlabel('Total control energy ∑‖u‖²')
axes[1].set_ylabel('Final state error')
axes[1].set_title('Energy-Accuracy Tradeoff Curve')

plt.suptitle('Pareto Frontier: State Error vs. Stimulation Energy', fontweight='bold')
plt.tight_layout()
plt.show()

print("This is Figure 3 of your paper (on real data).")
print("The Pareto frontier tells a clinician: 'to halve the state error, you need X more energy.'")

---
## 3. Applying LQR to Real Latent Trajectories

**Template for real data (after Module 4):**

1. **Define the target state**: mean latent position during successful 2-back maintenance (average `mu_np[mask_good, maint_period, :]`)

2. **Fit A from data**: use DMD on correct-trial maintenance trajectories

3. **Define B**: start with $B = I$ (stimulation in latent space). Later: use a learned mapping from electrode stimulation to latent space.

4. **Design LQR**: sweep R_c from 0.01 to 100, plot energy-accuracy curve

5. **Simulate rescue**: starting from a high-tangling pre-failure state, apply $u_t^* = -K z_t$, verify convergence

6. **The figure**: failing trajectory (gray) → LQR trajectory (blue) → converging to correct attractor (gold star)

---
## ✏️ Exercises

### A — Derive: What Does K Tell You?
The gain matrix K maps the current state error to a control action. For the toy 2D case:
1. Print K and explain what it means: which dimensions does it correct most aggressively?
2. Change Q_c to a diagonal matrix with different weights on each dimension. How does K change? Why?

### B — Stability Proof
By construction, LQR makes A - B@K stable (eigenvalues inside unit circle). Verify this numerically for 10 different (A, B, Q_c, R_c) combinations. Does it always hold? Under what conditions could it fail? (Hint: think about observability/controllability.)

### C — The Controllability Check
Before applying LQR, check if the system is controllable.
The controllability matrix: $\mathcal{R} = [B, AB, A^2B, \ldots, A^{n-1}B]$
If $\text{rank}(\mathcal{R}) < n$, some states are unreachable by control.

For your Miller data A and B = I: what is rank(ℝ)? For B = sparse (only some electrodes): which states become unreachable?

### D — The Clinical Question
A patient needs WM rescue. You have 4 ECoG electrodes over PFC. The B matrix (electrode → latent mapping) is estimated. The controllability matrix has rank 3 out of 8 latent dimensions. The 5 unreachable dimensions include the dimension that best predicts WM failure.

Write a paragraph: what are your options? What would you do?

*This is the kind of reasoning you need for a PhD application statement.*

---
## Next: `06_full_pipeline/06_figures_and_validation.ipynb`
## Write: `notes/lqr_riccati_derivation.md`